# Etapa 6 — Backtest e Avaliação

Simulação histórica da lógica de decisão (RSI + MACD + sentimento neutro fixo), sem chamadas de LLM, para os 4 tickers monitorados.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import yfinance as yf

from quantumfinance.backtesting.metrics import calculate_metrics
from quantumfinance.backtesting.strategy import run_backtest
from quantumfinance.features.targets import IBOVESPA_TICKER
from quantumfinance.universe import TICKERS

In [ ]:
BACKTEST_OUTPUT_PATH = Path("data/backtest_results.csv")

if BACKTEST_OUTPUT_PATH.exists():
    existing = pd.read_csv(BACKTEST_OUTPUT_PATH)
    print(f"AVISO: {BACKTEST_OUTPUT_PATH} já existe com {len(existing)} linha(s).")
    if not existing.empty:
        print(f"Período coberto: {existing['date'].min()} a {existing['date'].max()}")
    print("Rodar a próxima célula (run_backtest) vai SOBRESCREVER esse arquivo.")
    print("Se quiser preservar o resultado atual, copie o arquivo antes de continuar.")
else:
    print(f"{BACKTEST_OUTPUT_PATH} ainda não existe — a próxima célula vai criá-lo.")

In [ ]:
end_date = pd.Timestamp.today().strftime("%Y-%m-%d")
start_date = (pd.Timestamp.today() - pd.DateOffset(months=3)).strftime("%Y-%m-%d")

results = run_backtest(TICKERS, start_date, end_date, output_path=str(BACKTEST_OUTPUT_PATH))
print(f"Período: {start_date} a {end_date}")
print(f"Total de linhas: {len(results)}")
results.head()

In [ ]:
metrics = calculate_metrics(results)
for key, value in metrics.items():
    print(f"{key}: {value}")

## Gráfico 1 — Retorno acumulado: agente vs. buy-and-hold vs. Ibovespa

Para cada ticker, a posição do "agente" no dia é definida pela recomendação daquele dia
(COMPRAR = +1, VENDER = -1, AGUARDAR = 0), aplicada ao retorno diário real do ativo.

In [ ]:
def build_equity_curves(ticker: str, results: pd.DataFrame, start_date: str, end_date: str):
    """Constrói as séries de retorno acumulado: agente vs. buy-and-hold vs. Ibovespa."""
    position_map = {"COMPRAR": 1, "VENDER": -1, "AGUARDAR": 0}

    yf_ticker = f"{ticker}.SA"
    prices = yf.download(yf_ticker, start=start_date, end=end_date, progress=False, auto_adjust=True)
    if isinstance(prices.columns, pd.MultiIndex):
        prices.columns = prices.columns.get_level_values(0)
    daily_returns = prices["Close"].pct_change().fillna(0)

    ticker_results = results[results["ticker"] == ticker].set_index("date")

    positions = pd.Series(0.0, index=daily_returns.index)
    for d in daily_returns.index:
        date_str = d.strftime("%Y-%m-%d")
        if date_str in ticker_results.index:
            positions.loc[d] = position_map.get(ticker_results.loc[date_str, "recommendation"], 0)

    agent_curve = (1 + positions * daily_returns).cumprod()
    buy_hold_curve = (1 + daily_returns).cumprod()

    ibov_prices = yf.download(IBOVESPA_TICKER, start=start_date, end=end_date, progress=False, auto_adjust=True)
    if isinstance(ibov_prices.columns, pd.MultiIndex):
        ibov_prices.columns = ibov_prices.columns.get_level_values(0)
    ibov_returns = ibov_prices["Close"].pct_change().fillna(0)
    ibov_curve = (1 + ibov_returns).cumprod()

    return agent_curve, buy_hold_curve, ibov_curve


fig, axes = plt.subplots(2, 2, figsize=(14, 8))
for ax, ticker in zip(axes.flat, TICKERS):
    agent_curve, buy_hold_curve, ibov_curve = build_equity_curves(ticker, results, start_date, end_date)
    ax.plot(agent_curve.index, agent_curve.values, label="Agente")
    ax.plot(buy_hold_curve.index, buy_hold_curve.values, label="Buy-and-hold")
    ax.plot(ibov_curve.index, ibov_curve.values, label="Ibovespa")
    ax.set_title(ticker)
    ax.legend()
    ax.tick_params(axis="x", rotation=45)

fig.suptitle("Retorno acumulado: agente vs. buy-and-hold vs. Ibovespa")
plt.tight_layout()
plt.savefig("reports/figures/retorno_acumulado.png")
plt.show()

## Gráfico 2 — Distribuição das recomendações por ticker

In [ ]:
counts = results.groupby(["ticker", "recommendation"]).size().unstack(fill_value=0)
for col in ["COMPRAR", "VENDER", "AGUARDAR"]:
    if col not in counts.columns:
        counts[col] = 0
counts = counts[["COMPRAR", "VENDER", "AGUARDAR"]]

ax = counts.plot(kind="bar", stacked=True, figsize=(8, 5), color=["seagreen", "firebrick", "gray"])
ax.set_title("Distribuição das recomendações por ticker")
ax.set_xlabel("Ticker")
ax.set_ylabel("Número de dias")
plt.tight_layout()
plt.savefig("reports/figures/distribuicao_recomendacoes.png")
plt.show()

## Gráfico 3 — RSI médio por tipo de recomendação

In [ ]:
order = ["COMPRAR", "AGUARDAR", "VENDER"]
data_by_rec = [results.loc[results["recommendation"] == rec, "rsi"] for rec in order]

fig, ax = plt.subplots(figsize=(8, 5))
ax.boxplot(data_by_rec, tick_labels=order, showmeans=True)
ax.set_title("RSI médio por tipo de recomendação")
ax.set_ylabel("RSI")
plt.tight_layout()
plt.savefig("reports/figures/rsi_por_recomendacao.png")
plt.show()

## Limitação do sentimento histórico

O sentimento de notícias não está disponível para datas passadas via RSS.
O backtest usa sentiment_score=0.5 (neutro) para todas as datas históricas,
o que significa que os resultados refletem principalmente a qualidade dos
indicadores técnicos como sinais de decisão.